1. Configuración Inicial y Metadatos

In [32]:

"""
ANÁLISIS HÍBRIDO DE AUSENTISMO LABORAL - EQUIPO 15
Fecha: 25/09/2025
Versión: 2.0
Autor: Equipo de Análisis de Datos - Oriac Gimeno
Descripción: Análisis completo de factores que influyen en el ausentismo laboral
"""

'\nANÁLISIS HÍBRIDO DE AUSENTISMO LABORAL - EQUIPO 15\nFecha: 25/09/2025\nVersión: 2.0\nAutor: Equipo de Análisis de Datos - Oriac Gimeno\nDescripción: Análisis completo de factores que influyen en el ausentismo laboral\n'

2. Configuración de Entorno y Librerías

In [33]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro, spearmanr, kruskal, f_oneway, chi2_contingency, pointbiserialr, mannwhitneyu, ttest_ind
import warnings
import os
from datetime import datetime

# Configuración de estilo y warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuración de rutas
class Config:
    DATA_PATH = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"
    OUTPUT_BASE = r"G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Data\Resultados_Analisis_Frecuencia_Completo_220925"
    TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    @property
    def OUTPUT_PATH(self):
        return os.path.join(self.OUTPUT_BASE, f"Resultados_Analisis_Completo_{self.TIMESTAMP}")

config = Config()

3. Clases de Utilidades para Análisis Estadístico

In [34]:
class StatisticalAnalyzer:
    """Clase para realizar análisis estadísticos de manera estructurada"""
    
    @staticmethod
    def clasificar_correlacion(corr):
        """Clasificar la fuerza de la correlación según criterios estándar"""
        abs_corr = abs(corr)
        if abs_corr < 0.1:
            return "Muy débil", 1
        elif abs_corr < 0.3:
            return "Débil", 2
        elif abs_corr < 0.5:
            return "Moderada", 3
        elif abs_corr < 0.7:
            return "Fuerte", 4
        else:
            return "Muy fuerte", 5
    
    @staticmethod
    def cohens_d(x, y):
        """Calcular tamaño del efecto Cohen's d"""
        nx, ny = len(x), len(y)
        dof = nx + ny - 2
        pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
        return (np.mean(x) - np.mean(y)) / pooled_std
    
    @staticmethod
    def clasificar_efecto_cohen(d):
        """Clasificar el tamaño del efecto de Cohen"""
        abs_d = abs(d)
        if abs_d < 0.2:
            return "Muy pequeño"
        elif abs_d < 0.5:
            return "Pequeño"
        elif abs_d < 0.8:
            return "Mediano"
        else:
            return "Grande"

class DataValidator:
    """Clase para validaciones de datos"""
    
    @staticmethod
    def verificar_variables(df, variables_requeridas):
        """Verificar que las variables requeridas existen en el DataFrame"""
        faltantes = [var for var in variables_requeridas if var not in df.columns]
        if faltantes:
            raise ValueError(f"Variables faltantes en el dataset: {faltantes}")
        print("✓ Todas las variables requeridas están presentes")
        
    @staticmethod
    def verificar_tamanio_muestral(grupo1, grupo2, min_tamano=5):
        """Verificar tamaño muestral mínimo para análisis"""
        if len(grupo1) < min_tamano or len(grupo2) < min_tamano:
            return False
        return True

4. Carga y Validación de Datos

In [35]:

print("="*80)
print("FASE 1: CARGA Y PREPARACIÓN DE DATOS")
print("="*80)

try:
    # Cargar datos
    df = pd.read_parquet(config.DATA_PATH)
    print(f"✓ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    # Definir variables para análisis
    VARIABLES_NUMERICAS = [
        'Transportation_expense', 'Distance_Residence_Work', 'Service_time', 
        'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure', 
        'Son', 'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height', 
        'Body_mass_index', 'Education_numeric', 'Month_absence_order', 
        'Day_week_order', 'Seasons_order'
    ]
    
    VARIABLES_CATEGORICAS = ['Reason_absence', 'Month_absence', 'Day_week', 'Seasons', 'Education']
    TARGET_CONTINUO = 'Absenteeism_hours'
    
    # Validar variables
    DataValidator.verificar_variables(df, VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS + [TARGET_CONTINUO])
    
    # Preparar datos
    df_clean = df.drop(columns=['ID'], errors='ignore')
    median_absenteeism = df_clean[TARGET_CONTINUO].median()
    df_clean['Ausentismo_Alto'] = (df_clean[TARGET_CONTINUO] > median_absenteeism).astype(int)
    TARGET_BINARIO = 'Ausentismo_Alto'
    
    print(f"✓ Datos preparados. Mediana de ausentismo: {median_absenteeism:.2f} horas")
    print(f"✓ Variable binaria creada: {TARGET_BINARIO}")
    
except Exception as e:
    print(f"✗ Error en carga de datos: {e}")
    raise

FASE 1: CARGA Y PREPARACIÓN DE DATOS
✓ Dataset cargado: 806 filas, 25 columnas
✓ Todas las variables requeridas están presentes
✓ Datos preparados. Mediana de ausentismo: 3.00 horas
✓ Variable binaria creada: Ausentismo_Alto


4. 5. ANÁLISIS DE CALIDAD DE DATOS

In [39]:
# Celda 4.5: ANÁLISIS DE CALIDAD DE DATOS (VERSIÓN CORREGIDA)
print("\n" + "="*80)
print("FASE 1.5: ANÁLISIS DE CALIDAD DE DATOS")
print("="*80)

class AnalizadorCalidadDatos:
    """Clase para analizar la calidad de los datos antes del análisis principal"""
    
    def __init__(self, df):
        self.df = df
        self.problemas_encontrados = []
    
    def analizar_valores_cero_problematicos(self):
        """Analizar valores cero que podrían ser problemáticos"""
        print("🔍 Analizando valores cero problemáticos...")
        
        # Variables donde el valor 0 podría ser problemático
        variables_problematicas = {
            'Reason_absence': 'Razón de ausencia (0 podría ser error)',
            'Month_absence': 'Mes de ausencia (0 podría ser error)',
            'Absenteeism_hours': 'Horas de ausencia (0 podría ser registro sin ausencia)',
            'Hit_target': 'Porcentaje de cumplimiento de metas (0 podría ser error)'
        }
        
        resultados = {}
        
        for variable, descripcion in variables_problematicas.items():
            if variable in self.df.columns:
                # Convertir a numérico si es necesario
                if self.df[variable].dtype == 'object':
                    try:
                        serie_numerica = pd.to_numeric(self.df[variable], errors='coerce')
                        conteo_cero = (serie_numerica == 0).sum()
                    except:
                        conteo_cero = 0
                else:
                    conteo_cero = (self.df[variable] == 0).sum()
                
                total_registros = len(self.df)
                porcentaje = (conteo_cero / total_registros) * 100
                
                resultados[variable] = {
                    'descripcion': descripcion,
                    'conteo_cero': conteo_cero,
                    'total_registros': total_registros,
                    'porcentaje': round(porcentaje, 2),
                    'tipo_dato': str(self.df[variable].dtype)
                }
                
                print(f"  {variable}: {conteo_cero} registros con valor 0 ({porcentaje:.2f}%) - Tipo: {self.df[variable].dtype}")
                
                if conteo_cero > 0:
                    # Mostrar ejemplos de registros problemáticos
                    if variable in ['Reason_absence', 'Month_absence']:
                        # Para estas variables, buscar valores que sean 0 o '0'
                        if self.df[variable].dtype == 'object':
                            registros_cero = self.df[self.df[variable].astype(str) == '0'].head(3)
                        else:
                            registros_cero = self.df[self.df[variable] == 0].head(3)
                    else:
                        if self.df[variable].dtype == 'object':
                            registros_cero = self.df[pd.to_numeric(self.df[variable], errors='coerce') == 0].head(3)
                        else:
                            registros_cero = self.df[self.df[variable] == 0].head(3)
                    
                    if len(registros_cero) > 0:
                        print(f"    Ejemplos de registros con {variable} = 0:")
                        for idx, row in registros_cero.iterrows():
                            print(f"      ID {row.get('ID', 'N/A')}, "
                                  f"Mes {row.get('Month_absence', 'N/A')}, "
                                  f"Razón {row.get('Reason_absence', 'N/A')}, "
                                  f"Horas: {row.get('Absenteeism_hours', 'N/A')}")
        
        return resultados
    
    def analizar_valores_frecuentes_cero(self):
        """Analizar variables que frecuentemente tienen valor 0"""
        print("\n🔍 Analizando variables con valores 0 frecuentes...")
        
        variables_cero_frecuente = [
            'Disciplinary_failure', 'Social_drinker', 'Social_smoker', 'Pet'
        ]
        
        resultados = {}
        
        for variable in variables_cero_frecuente:
            if variable in self.df.columns:
                # Manejar tipos de datos tanto numéricos como strings
                if self.df[variable].dtype == 'object':
                    try:
                        serie_numerica = pd.to_numeric(self.df[variable], errors='coerce')
                        conteo_cero = (serie_numerica == 0).sum()
                    except:
                        conteo_cero = 0
                else:
                    conteo_cero = (self.df[variable] == 0).sum()
                
                total_registros = len(self.df)
                porcentaje = (conteo_cero / total_registros) * 100
                
                resultados[variable] = {
                    'conteo_cero': conteo_cero,
                    'total_registros': total_registros,
                    'porcentaje': round(porcentaje, 2),
                    'tipo_dato': str(self.df[variable].dtype)
                }
                
                print(f"  {variable}: {conteo_cero} registros con valor 0 ({porcentaje:.2f}%) - Tipo: {self.df[variable].dtype}")
                
                # Determinar si es problemático
                if porcentaje > 90:
                    self.problemas_encontrados.append(f"⚠ {variable} tiene {porcentaje}% de valores 0 (posible desbalance)")
                elif porcentaje > 70:
                    self.problemas_encontrados.append(f"ℹ {variable} tiene {porcentaje}% de valores 0 (alta frecuencia)")
        
        return resultados
    
    def analizar_inconsistencias_temporales(self):
        """Analizar inconsistencias en variables temporales"""
        print("\n🔍 Analizando inconsistencias temporales...")
        
        inconsistencias = []
        
        # Verificar meses válidos (1-12 o nombres de meses)
        if 'Month_absence' in self.df.columns:
            # Determinar el tipo de datos de Month_absence
            if self.df['Month_absence'].dtype == 'object':
                # Es string - verificar nombres de meses válidos
                meses_validos = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
                               'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre',
                               'January', 'February', 'March', 'April', 'May', 'June',
                               'July', 'August', 'September', 'October', 'November', 'December',
                               '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
                
                # Convertir a minúsculas para comparación case-insensitive
                meses_lower = self.df['Month_absence'].astype(str).str.lower()
                meses_invalidos = self.df[~meses_lower.isin([m.lower() for m in meses_validos])]
                
                if len(meses_invalidos) > 0:
                    print(f"  ❌ {len(meses_invalidos)} registros con meses inválidos (strings)")
                    print(f"    Valores únicos encontrados: {meses_invalidos['Month_absence'].unique()}")
                    inconsistencias.append(f"Meses inválidos (strings): {len(meses_invalidos)} registros")
            else:
                # Es numérico - verificar rango 1-12
                try:
                    meses_invalidos = self.df[~self.df['Month_absence'].between(1, 12)]
                    if len(meses_invalidos) > 0:
                        print(f"  ❌ {len(meses_invalidos)} registros con meses inválidos (numéricos)")
                        inconsistencias.append(f"Meses inválidos (numéricos): {len(meses_invalidos)} registros")
                except TypeError as e:
                    print(f"  ⚠ Error al verificar meses numéricos: {e}")
        
        # Verificar días de la semana válidos (1-7 si existe)
        if 'Day_week' in self.df.columns:
            try:
                if self.df['Day_week'].dtype == 'object':
                    # Convertir a numérico para verificación
                    dias_numericos = pd.to_numeric(self.df['Day_week'], errors='coerce')
                    dias_invalidos = self.df[dias_numericos.isna() | ~dias_numericos.between(1, 7)]
                else:
                    dias_invalidos = self.df[~self.df['Day_week'].between(1, 7)]
                
                if len(dias_invalidos) > 0:
                    print(f"  ❌ {len(dias_invalidos)} registros con días de semana inválidos")
                    inconsistencias.append(f"Días de semana inválidos: {len(dias_invalidos)} registros")
            except Exception as e:
                print(f"  ⚠ Error al verificar días de semana: {e}")
        
        return inconsistencias
    
    def analizar_ausentismo_cero(self):
        """Análisis específico de registros con horas de ausencia = 0"""
        print("\n🔍 Analizando registros con ausentismo = 0 horas...")
        
        if 'Absenteeism_hours' in self.df.columns:
            # Manejar diferentes tipos de datos
            if self.df['Absenteeism_hours'].dtype == 'object':
                try:
                    ausentismo_numerico = pd.to_numeric(self.df['Absenteeism_hours'], errors='coerce')
                    ausentismo_cero = self.df[ausentismo_numerico == 0]
                except:
                    ausentismo_cero = pd.DataFrame()
            else:
                ausentismo_cero = self.df[self.df['Absenteeism_hours'] == 0]
            
            total_registros = len(self.df)
            conteo_cero = len(ausentismo_cero)
            porcentaje = (conteo_cero / total_registros) * 100
            
            print(f"  Registros con Ausentismo = 0 horas: {conteo_cero} ({porcentaje:.2f}%)")
            
            if conteo_cero > 0:
                print("  Ejemplos de registros con 0 horas de ausentismo:")
                ejemplos = ausentismo_cero.head(3)
                for idx, row in ejemplos.iterrows():
                    print(f"    ID {row.get('ID', 'N/A')}, "
                          f"Razón: {row.get('Reason_absence', 'N/A')}, "
                          f"Mes: {row.get('Month_absence', 'N/A')}")
            
            return {
                'conteo_cero': conteo_cero,
                'total_registros': total_registros,
                'porcentaje': round(porcentaje, 2)
            }
        
        return None
    
    def generar_recomendaciones(self, resultados_cero, resultados_frecuentes, resultados_ausentismo_cero):
        """Generar recomendaciones basadas en los análisis"""
        print("\n" + "="*50)
        print("RECOMENDACIONES DE LIMPIEZA DE DATOS")
        print("="*50)
        
        recomendaciones = []
        
        # Análisis de valores cero problemáticos
        for variable, info in resultados_cero.items():
            if info['conteo_cero'] > 0:
                if variable == 'Reason_absence':
                    recomendaciones.append(
                        f"❌ ELIMINAR registros con {variable}=0 "
                        f"({info['conteo_cero']} registros, {info['porcentaje']}%) - "
                        f"Razón de ausencia 0 probablemente es error de datos"
                    )
                elif variable == 'Month_absence':
                    # Solo recomendar eliminar si son valores numéricos 0
                    if info['tipo_dato'] != 'object' or '0' in self.df['Month_absence'].astype(str).values:
                        recomendaciones.append(
                            f"❌ ELIMINAR registros con {variable}=0 "
                            f"({info['conteo_cero']} registros, {info['porcentaje']}%) - "
                            f"Mes 0 no es válido"
                        )
                elif variable == 'Hit_target' and info['porcentaje'] > 10:
                    recomendaciones.append(
                        f"⚠ REVISAR registros con {variable}=0 "
                        f"({info['conteo_cero']} registros, {info['porcentaje']}%) - "
                        f"Podría indicar problemas de rendimiento"
                    )
        
        # Análisis de ausentismo cero
        if resultados_ausentismo_cero and resultados_ausentismo_cero['porcentaje'] > 20:
            recomendaciones.append(
                f"⚠ CONSIDERAR eliminar registros con Ausentismo=0 "
                f"({resultados_ausentismo_cero['conteo_cero']} registros, {resultados_ausentismo_cero['porcentaje']}%) - "
                f"No representan casos de ausentismo real"
            )
        
        # Mostrar recomendaciones
        if recomendaciones:
            for i, recomendacion in enumerate(recomendaciones, 1):
                print(f"{i}. {recomendacion}")
        else:
            print("✅ No se encontraron problemas críticos que requieran eliminación de datos")
        
        return recomendaciones
    
    def aplicar_limpieza(self, recomendaciones, eliminar_ausentismo_cero=False):
        """Aplicar limpieza basada en las recomendaciones"""
        print("\n" + "="*50)
        print("APLICANDO LIMPIEZA DE DATOS")
        print("="*50)
        
        df_limpio = self.df.copy()
        registros_eliminados = 0
        total_inicial = len(df_limpio)
        
        # Eliminar registros con Reason_absence = 0 o '0'
        if 'Reason_absence' in df_limpio.columns:
            if df_limpio['Reason_absence'].dtype == 'object':
                mask_razon_cero = df_limpio['Reason_absence'].astype(str) == '0'
            else:
                mask_razon_cero = df_limpio['Reason_absence'] == 0
            
            if mask_razon_cero.any():
                eliminados = mask_razon_cero.sum()
                df_limpio = df_limpio[~mask_razon_cero]
                registros_eliminados += eliminados
                print(f"✓ Eliminados {eliminados} registros con Reason_absence = 0")
        
        # Eliminar registros con Month_absence = 0 o '0'
        if 'Month_absence' in df_limpio.columns:
            if df_limpio['Month_absence'].dtype == 'object':
                mask_mes_cero = df_limpio['Month_absence'].astype(str) == '0'
            else:
                mask_mes_cero = df_limpio['Month_absence'] == 0
            
            if mask_mes_cero.any():
                eliminados = mask_mes_cero.sum()
                df_limpio = df_limpio[~mask_mes_cero]
                registros_eliminados += eliminados
                print(f"✓ Eliminados {eliminados} registros con Month_absence = 0")
        
        # Eliminar registros con Absenteeism_hours = 0 (opcional)
        if eliminar_ausentismo_cero and 'Absenteeism_hours' in df_limpio.columns:
            if df_limpio['Absenteeism_hours'].dtype == 'object':
                ausentismo_numerico = pd.to_numeric(df_limpio['Absenteeism_hours'], errors='coerce')
                mask_ausentismo_cero = ausentismo_numerico == 0
            else:
                mask_ausentismo_cero = df_limpio['Absenteeism_hours'] == 0
            
            if mask_ausentismo_cero.any():
                eliminados = mask_ausentismo_cero.sum()
                df_limpio = df_limpio[~mask_ausentismo_cero]
                registros_eliminados += eliminados
                print(f"✓ Eliminados {eliminados} registros con Absenteeism_hours = 0")
        
        # Estadísticas finales
        total_final = len(df_limpio)
        porcentaje_eliminado = (registros_eliminados / total_inicial) * 100
        
        print(f"\n📊 RESUMEN DE LIMPIEZA:")
        print(f"  Registros iniciales: {total_inicial}")
        print(f"  Registros eliminados: {registros_eliminados}")
        print(f"  Registros finales: {total_final}")
        print(f"  Porcentaje eliminado: {porcentaje_eliminado:.2f}%")
        
        if porcentaje_eliminado > 20:
            print("⚠ ADVERTENCIA: Se eliminó más del 20% de los datos")
        
        return df_limpio

# Ejecutar análisis de calidad
analizador_calidad = AnalizadorCalidadDatos(df)

# 1. Analizar valores cero problemáticos
resultados_cero = analizador_calidad.analizar_valores_cero_problematicos()

# 2. Analizar valores cero frecuentes
resultados_frecuentes = analizador_calidad.analizar_valores_frecuentes_cero()

# 3. Analizar inconsistencias temporales
inconsistencias = analizador_calidad.analizar_inconsistencias_temporales()

# 4. Análisis específico de ausentismo cero
resultados_ausentismo_cero = analizador_calidad.analizar_ausentismo_cero()

# 5. Generar recomendaciones
recomendaciones = analizador_calidad.generar_recomendaciones(
    resultados_cero, resultados_frecuentes, resultados_ausentismo_cero
)

# 6. Preguntar al usuario qué acción tomar
print("\n" + "="*50)
print("DECISIÓN DE LIMPIEZA DE DATOS")
print("="*50)

# Mostrar opciones al usuario
print("Opciones disponibles:")
print("1. Eliminar solo registros con Reason_absence=0 y Month_absence=0 (RECOMENDADO)")
print("2. Eliminar registros con ausentismo=0 además de los anteriores")
print("3. No eliminar ningún registro (conservar todos los datos)")
print("4. Eliminar registros con ausentismo=0 solamente")

try:
    opcion = int(input("\nSeleccione una opción (1-4): "))
    
    if opcion == 1:
        df_clean = analizador_calidad.aplicar_limpieza(recomendaciones, eliminar_ausentismo_cero=False)
        print("✅ Aplicada limpieza básica (Reason_absence=0 y Month_absence=0)")
    elif opcion == 2:
        df_clean = analizador_calidad.aplicar_limpieza(recomendaciones, eliminar_ausentismo_cero=True)
        print("✅ Aplicada limpieza completa (incluyendo Ausentismo=0)")
    elif opcion == 3:
        df_clean = df.copy()
        print("✅ Se conservaron todos los registros (sin limpieza)")
    elif opcion == 4:
        # Limpieza personalizada: solo ausentismo=0
        df_clean = df.copy()
        if 'Absenteeism_hours' in df_clean.columns:
            if df_clean['Absenteeism_hours'].dtype == 'object':
                ausentismo_numerico = pd.to_numeric(df_clean['Absenteeism_hours'], errors='coerce')
                mask_ausentismo_cero = ausentismo_numerico == 0
            else:
                mask_ausentismo_cero = df_clean['Absenteeism_hours'] == 0
            
            if mask_ausentismo_cero.any():
                eliminados = mask_ausentismo_cero.sum()
                df_clean = df_clean[~mask_ausentismo_cero]
                print(f"✅ Eliminados {eliminados} registros con Ausentismo=0")
    else:
        print("❌ Opción inválida. Se aplicará limpieza básica por defecto.")
        df_clean = analizador_calidad.aplicar_limpieza(recomendaciones, eliminar_ausentismo_cero=False)

except ValueError:
    print("❌ Entrada inválida. Se aplicará limpieza básica por defecto.")
    df_clean = analizador_calidad.aplicar_limpieza(recomendaciones, eliminar_ausentismo_cero=False)

# Continuar con el análisis normal...
print(f"\n✅ Dataset listo para análisis: {df_clean.shape[0]} filas, {df_clean.shape[1]} columnas")

# Actualizar las variables para el análisis principal
median_absenteeism = df_clean['Absenteeism_hours'].median()
df_clean['Ausentismo_Alto'] = (df_clean['Absenteeism_hours'] > median_absenteeism).astype(int)
TARGET_BINARIO = 'Ausentismo_Alto'

print(f"✓ Variable binaria creada: {TARGET_BINARIO} (mediana: {median_absenteeism:.2f} horas)")


FASE 1.5: ANÁLISIS DE CALIDAD DE DATOS
🔍 Analizando valores cero problemáticos...
  Reason_absence: 0 registros con valor 0 (0.00%) - Tipo: string
  Month_absence: 0 registros con valor 0 (0.00%) - Tipo: string
  Absenteeism_hours: 48 registros con valor 0 (5.96%) - Tipo: float64
    Ejemplos de registros con Absenteeism_hours = 0:
      ID 36, Mes Julio, Razón 0, Horas: 0.0
      ID 20, Mes Septiembre, Razón 0, Horas: 0.0
      ID 29, Mes Septiembre, Razón 0, Horas: 0.0
  Hit_target: 0 registros con valor 0 (0.00%) - Tipo: float64

🔍 Analizando variables con valores 0 frecuentes...
  Disciplinary_failure: 764 registros con valor 0 (94.79%) - Tipo: int64
  Social_drinker: 353 registros con valor 0 (43.80%) - Tipo: int64
  Social_smoker: 742 registros con valor 0 (92.06%) - Tipo: int64
  Pet: 492 registros con valor 0 (61.04%) - Tipo: int64

🔍 Analizando inconsistencias temporales...
  ⚠ Error al verificar meses numéricos: '>=' not supported between instances of 'str' and 'int'
  ⚠ Err

5. Análisis de Normalidad

In [40]:

print("\n" + "="*80)
print("FASE 2: ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)")
print("="*80)

def analizar_normalidad(df, variables):
    """Realizar test de normalidad Shapiro-Wilk para múltiples variables"""
    resultados = []
    
    for var in variables:
        data = df[var].dropna()
        if len(data) > 3:  # Mínimo requerido para Shapiro-Wilk
            stat, p_value = shapiro(data)
            resultados.append({
                'Variable': var,
                'Estadistico_Shapiro': round(stat, 6),
                'p_value_Shapiro': round(p_value, 6),
                'Es_Normal': p_value > 0.05,
                'n': len(data)
            })
        else:
            print(f"⚠ Variable {var} no tiene suficientes datos para análisis de normalidad")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_normality = analizar_normalidad(df_clean, VARIABLES_NUMERICAS + [TARGET_CONTINUO])
print("Resultados de normalidad:")
print(df_normality.to_string(index=False))

# Resumen estadístico
normales = df_normality['Es_Normal'].sum()
total_vars = len(df_normality)
print(f"\n📊 Resumen: {normales}/{total_vars} variables siguen distribución normal (p > 0.05)")


FASE 2: ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)
Resultados de normalidad:
               Variable  Estadistico_Shapiro  p_value_Shapiro  Es_Normal   n
 Transportation_expense             0.947629              0.0      False 806
Distance_Residence_Work             0.880630              0.0      False 806
           Service_time             0.943835              0.0      False 806
                    Age             0.928054              0.0      False 806
  Work_load_Average_day             0.923185              0.0      False 806
             Hit_target             0.887959              0.0      False 806
   Disciplinary_failure             0.229883              0.0      False 806
                    Son             0.817792              0.0      False 806
         Social_drinker             0.630958              0.0      False 806
          Social_smoker             0.298818              0.0      False 806
                    Pet             0.620236              0.0      False 806
   

6. Análisis de Variables Numéricas

In [41]:

print("\n" + "="*80)
print("FASE 3: ANÁLISIS DE VARIABLES NUMÉRICAS")
print("="*80)

def analizar_variables_numericas(df, variables, target_cont, target_bin, df_normality):
    """Análisis completo para variables numéricas"""
    resultados = []
    analyzer = StatisticalAnalyzer()
    
    for i, var in enumerate(variables, 1):
        print(f"🔍 Analizando {i}/{len(variables)}: {var}")
        
        if var not in df.columns:
            print(f"  ⚠ Variable {var} no encontrada, saltando...")
            continue
            
        # Preparar datos para grupos
        data_alto = df[df[target_bin] == 1][var].dropna()
        data_bajo = df[df[target_bin] == 0][var].dropna()
        
        # Verificar tamaño muestral
        if not DataValidator.verificar_tamanio_muestral(data_alto, data_bajo):
            print(f"  ⚠ Tamaño muestral insuficiente para {var}, saltando...")
            continue
        
        try:
            # Correlación Spearman
            spearman_corr, spearman_p = spearmanr(df[var].dropna(), df[target_cont].dropna())
            fuerza_spearman, fuerza_num = analyzer.clasificar_correlacion(spearman_corr)
            
            # Point-biserial
            pointbiserial_corr, pointbiserial_p = pointbiserialr(df[var].dropna(), df[target_bin].dropna())
            
            # Tests de diferencias
            mw_stat, mw_p = mannwhitneyu(data_alto, data_bajo, alternative='two-sided')
            t_stat, t_p = ttest_ind(data_alto, data_bajo, equal_var=False)
            
            # Tamaño del efecto
            d_effect = analyzer.cohens_d(data_alto, data_bajo)
            tamaño_efecto = analyzer.clasificar_efecto_cohen(d_effect)
            
            # Determinar test apropiado
            es_normal_var = df_normality[df_normality['Variable'] == var]['Es_Normal'].values[0]
            test_medias = "T-test" if es_normal_var else "Mann-Whitney"
            p_value_medias = t_p if es_normal_var else mw_p
            
            resultados.append({
                'Variable': var,
                'Correlacion_Spearman': round(spearman_corr, 6),
                'p_value_Spearman': round(spearman_p, 6),
                'Fuerza_Spearman': fuerza_spearman,
                'Significativa_Spearman': spearman_p < 0.05,
                'Cohens_d': round(d_effect, 6),
                'Tamaño_Efecto': tamaño_efecto,
                'Test_Medias': test_medias,
                'p_value_Medias': round(p_value_medias, 6),
                'Significativa_Medias': p_value_medias < 0.05,
                'Media_Alto': round(data_alto.mean(), 3),
                'Media_Bajo': round(data_bajo.mean(), 3),
                'Diferencia_Medias': round(data_alto.mean() - data_bajo.mean(), 3),
                'abs_corr': abs(spearman_corr),
                'abs_effect': abs(d_effect)
            })
            
        except Exception as e:
            print(f"  ✗ Error analizando {var}: {e}")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_results_numeric = analizar_variables_numericas(
    df_clean, VARIABLES_NUMERICAS, TARGET_CONTINUO, TARGET_BINARIO, df_normality
)

print("\n📋 Resultados del análisis numérico:")
print(df_results_numeric[['Variable', 'Correlacion_Spearman', 'Fuerza_Spearman', 
                         'Significativa_Spearman', 'Cohens_d', 'Tamaño_Efecto']].to_string(index=False))


FASE 3: ANÁLISIS DE VARIABLES NUMÉRICAS
🔍 Analizando 1/18: Transportation_expense
🔍 Analizando 2/18: Distance_Residence_Work
🔍 Analizando 3/18: Service_time
🔍 Analizando 4/18: Age
🔍 Analizando 5/18: Work_load_Average_day
🔍 Analizando 6/18: Hit_target
🔍 Analizando 7/18: Disciplinary_failure
🔍 Analizando 8/18: Son
🔍 Analizando 9/18: Social_drinker
🔍 Analizando 10/18: Social_smoker
🔍 Analizando 11/18: Pet
🔍 Analizando 12/18: Weight
🔍 Analizando 13/18: Height
🔍 Analizando 14/18: Body_mass_index
🔍 Analizando 15/18: Education_numeric
🔍 Analizando 16/18: Month_absence_order
🔍 Analizando 17/18: Day_week_order
🔍 Analizando 18/18: Seasons_order

📋 Resultados del análisis numérico:
               Variable  Correlacion_Spearman Fuerza_Spearman  Significativa_Spearman  Cohens_d Tamaño_Efecto
 Transportation_expense              0.170418           Débil                    True  0.496882       Pequeño
Distance_Residence_Work              0.016524       Muy débil                   False  0.071608   M

7. Análisis de Variables Categóricas

In [42]:

print("\n" + "="*80)
print("FASE 4: ANÁLISIS DE VARIABLES CATEGÓRICAS")
print("="*80)

def analizar_variables_categoricas(df, variables, target_cont, target_bin, df_normality):
    """Análisis completo para variables categóricas"""
    resultados = []
    
    for i, var in enumerate(variables, 1):
        print(f"🔍 Analizando {i}/{len(variables)}: {var}")
        
        if var not in df.columns:
            print(f"  ⚠ Variable {var} no encontrada, saltando...")
            continue
            
        try:
            # Kruskal-Wallis
            grupos_kw = [grupo[target_cont].values for nombre, grupo in df.groupby(var) if len(grupo) > 5]
            
            if len(grupos_kw) > 1:
                kw_stat, kw_p = kruskal(*grupos_kw)
            else:
                kw_stat, kw_p = np.nan, np.nan
                
            # Chi-cuadrado
            tabla_contingencia = pd.crosstab(df[var], df[target_bin])
            if tabla_contingencia.shape[0] > 1 and tabla_contingencia.shape[1] > 1:
                chi2_stat, chi2_p, dof, esperado = chi2_contingency(tabla_contingencia)
            else:
                chi2_stat, chi2_p = np.nan, np.nan
            
            # Determinar test principal
            es_normal_target = df_normality[df_normality['Variable'] == target_cont]['Es_Normal'].values[0]
            test_principal = 'ANOVA' if es_normal_target else 'Kruskal-Wallis'
            p_value_principal = kw_p
            
            resultados.append({
                'Variable': var,
                'Test_Principal': test_principal,
                'p_value_Principal': round(p_value_principal, 6) if not np.isnan(p_value_principal) else np.nan,
                'Significativa_Principal': p_value_principal < 0.05 if not np.isnan(p_value_principal) else False,
                'p_value_Chi2': round(chi2_p, 6) if not np.isnan(chi2_p) else np.nan,
                'Significativa_Chi2': chi2_p < 0.05 if not np.isnan(chi2_p) else False,
                'n_Categorias': len(df[var].unique())
            })
            
        except Exception as e:
            print(f"  ✗ Error analizando {var}: {e}")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_results_categorical = analizar_variables_categoricas(
    df_clean, VARIABLES_CATEGORICAS, TARGET_CONTINUO, TARGET_BINARIO, df_normality
)

print("\n📋 Resultados del análisis categórico:")
print(df_results_categorical[['Variable', 'Test_Principal', 'p_value_Principal', 
                            'Significativa_Principal', 'Significativa_Chi2']].to_string(index=False))


FASE 4: ANÁLISIS DE VARIABLES CATEGÓRICAS
🔍 Analizando 1/5: Reason_absence
🔍 Analizando 2/5: Month_absence
🔍 Analizando 3/5: Day_week
🔍 Analizando 4/5: Seasons
🔍 Analizando 5/5: Education

📋 Resultados del análisis categórico:
      Variable Test_Principal  p_value_Principal  Significativa_Principal  Significativa_Chi2
Reason_absence Kruskal-Wallis           0.000000                     True                True
 Month_absence Kruskal-Wallis           0.001513                     True                True
      Day_week Kruskal-Wallis           0.020356                     True                True
       Seasons Kruskal-Wallis           0.021905                     True                True
     Education Kruskal-Wallis           0.423551                    False               False


8. Visualizaciones (Modularizado)

In [43]:

print("\n" + "="*80)
print("FASE 5: GENERACIÓN DE VISUALIZACIONES")
print("="*80)

class VisualizacionGenerator:
    """Generador de visualizaciones para el análisis"""
    
    def __init__(self, output_path):
        self.output_path = output_path
        os.makedirs(output_path, exist_ok=True)
    
    def generar_heatmap_correlaciones(self, df, variables, target):
        """Generar heatmap de correlaciones Spearman"""
        print("📊 Generando heatmap de correlaciones...")
        
        corr_vars = [v for v in variables if v in df.columns] + [target]
        if len(corr_vars) > 1:
            corr_matrix = df[corr_vars].corr(method='spearman')
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            
            plt.figure(figsize=(16, 14))
            sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                       square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
            plt.title('MATRIZ DE CORRELACIÓN SPEARMAN - AUSENTISMO LABORAL\n(Todas las variables numéricas)', 
                     fontsize=16, fontweight='bold', pad=20)
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/1_heatmap_correlaciones.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Heatmap de correlaciones guardado")
        else:
            print("⚠ No hay suficientes variables para generar heatmap")
    
    def generar_importancia_variables(self, df_results):
        """Gráfico de importancia de variables"""
        print("📊 Generando gráfico de importancia...")
        
        if df_results.empty:
            print("⚠ No hay datos para generar gráfico de importancia")
            return
            
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
        
        # Importancia por correlación
        corr_data = df_results.sort_values('abs_corr', ascending=True)
        colors = ['red' if sig else 'gray' for sig in corr_data['Significativa_Spearman']]
        ax1.barh(corr_data['Variable'], corr_data['abs_corr'], color=colors, alpha=0.7)
        ax1.set_xlabel('Correlación Absoluta (Spearman)')
        ax1.set_title('IMPORTANCIA POR CORRELACIÓN\n(Rojo = Significativo, p < 0.05)', fontweight='bold')
        ax1.grid(axis='x', alpha=0.3)
        
        # Importancia por tamaño del efecto
        effect_data = df_results.sort_values('abs_effect', ascending=True)
        colors_effect = ['red' if sig else 'gray' for sig in effect_data['Significativa_Medias']]
        ax2.barh(effect_data['Variable'], effect_data['abs_effect'], color=colors_effect, alpha=0.7)
        ax2.set_xlabel('Tamaño del Efecto Absoluto (Cohen\'s d)')
        ax2.set_title('IMPORTANCIA POR TAMAÑO DEL EFECTO\n(Rojo = Diferencias Significativas)', fontweight='bold')
        ax2.grid(axis='x', alpha=0.3)
        
        plt.suptitle('COMPARACIÓN DE IMPORTANCIA DE VARIABLES', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{self.output_path}/2_importancia_variables.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Gráfico de importancia de variables guardado")
    
    def generar_scatter_variables_significativas(self, df, df_results, target, max_variables=6):
        """Scatter plots para variables más significativas"""
        print("📊 Generando scatter plots de variables significativas...")
        
        if df_results.empty:
            print("⚠ No hay resultados para generar scatter plots")
            return
            
        sig_vars_df = df_results[df_results['Significativa_Spearman']]
        if sig_vars_df.empty:
            print("⚠ No hay variables significativas para scatter plots")
            return
            
        sig_vars = sig_vars_df.nlargest(max_variables, 'abs_corr')['Variable'].tolist()
        n_vars = len(sig_vars)
        
        if n_vars == 0:
            print("⚠ No hay variables significativas para mostrar")
            return
        
        n_cols = min(3, n_vars)
        n_rows = (n_vars + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        
        # Aplanar axes para facilitar el acceso
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.ravel()
        
        for i, var in enumerate(sig_vars):
            if i < len(axes):
                sns.regplot(data=df, x=var, y=target, ax=axes[i], 
                           scatter_kws={'alpha':0.6, 's':30}, line_kws={'color':'red'}, ci=95)
                
                stats_row = df_results[df_results['Variable'] == var].iloc[0]
                corr = stats_row['Correlacion_Spearman']
                p_val = stats_row['p_value_Spearman']
                fuerza = stats_row['Fuerza_Spearman']
                direccion = "Positiva" if corr > 0 else "Negativa"
                
                axes[i].set_title(f'{var}\nρ = {corr:.3f} ({fuerza}, {direccion})', 
                                fontweight='bold', fontsize=10)
                axes[i].set_xlabel(var)
                axes[i].set_ylabel('Horas de Ausentismo')
        
        # Ocultar ejes vacíos
        for j in range(len(sig_vars), n_rows * n_cols):
            if j < len(axes):
                axes[j].set_visible(False)
        
        plt.suptitle('RELACIÓN DE VARIABLES MÁS SIGNIFICATIVAS CON AUSENTISMO', 
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{self.output_path}/3_scatter_variables_significativas.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Scatter plots de variables significativas guardados")
    
    def generar_boxplots_categoricas(self, df, df_results, target):
        """Boxplots para variables categóricas significativas"""
        print("📊 Generando boxplots de variables categóricas...")
        
        if df_results.empty:
            print("⚠ No hay resultados categóricos para generar boxplots")
            return
            
        sig_cat_vars = df_results[df_results['Significativa_Principal']]['Variable'].tolist()
        
        if not sig_cat_vars:
            print("⚠ No hay variables categóricas significativas")
            return
        
        n_cat_vars = len(sig_cat_vars)
        n_cols = min(2, n_cat_vars)
        n_rows = (n_cat_vars + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
        
        # Aplanar axes para facilitar el acceso
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.ravel()
        
        for i, cat_var in enumerate(sig_cat_vars):
            if i < len(axes):
                order = df.groupby(cat_var)[target].median().sort_values(ascending=False).index
                
                sns.boxplot(data=df, x=cat_var, y=target, ax=axes[i], order=order)
                axes[i].set_title(f'Ausentismo por {cat_var}', fontweight='bold')
                axes[i].tick_params(axis='x', rotation=45)
                axes[i].set_xlabel(cat_var)
                axes[i].set_ylabel('Horas de Ausentismo')
        
        # Ocultar ejes vacíos
        for j in range(len(sig_cat_vars), n_rows * n_cols):
            if j < len(axes):
                axes[j].set_visible(False)
        
        plt.suptitle('DISTRIBUCIÓN DE AUSENTISMO POR VARIABLES CATEGÓRICAS SIGNIFICATIVAS', 
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{self.output_path}/4_boxplots_categoricas.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Boxplots de variables categóricas guardados")
    
    def generar_grafico_correlaciones_impacto(self, df_results):
        """Gráfico de correlaciones mejorado - fuerza y dirección"""
        print("📊 Generando gráfico de correlaciones de impacto...")
        
        if df_results.empty:
            print("⚠ No hay resultados para generar gráfico de impacto")
            return
            
        df_plot = df_results.sort_values('abs_corr', ascending=True)
        
        plt.figure(figsize=(14, 10))
        
        # Definir colores basados en dirección y fuerza
        colors = []
        for _, row in df_plot.iterrows():
            if row['Correlacion_Spearman'] > 0:
                intensity = min(0.3 + row['Fuerza_Num'] * 0.15, 0.9)
                colors.append((intensity, 0.2, 0.2, 0.8))
            else:
                intensity = min(0.3 + row['Fuerza_Num'] * 0.15, 0.9)
                colors.append((0.2, 0.2, intensity, 0.8))
        
        # Crear gráfico de barras
        bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], color=colors, alpha=0.8)
        
        # Añadir etiquetas de valores
        for i, (value, variable) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['Variable'])):
            if value >= 0:
                plt.text(value + 0.01, i, f'{value:.3f}', va='center', ha='left', fontweight='bold')
            else:
                plt.text(value - 0.01, i, f'{value:.3f}', va='center', ha='right', fontweight='bold')
        
        # Línea vertical en 0
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
        
        # Personalizar
        plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=12, fontweight='bold')
        plt.title('IMPACTO DE VARIABLES EN EL AUSENTISMO LABORAL\n(Fuerza y Dirección de Correlación)', 
                  fontsize=16, fontweight='bold', pad=20)
        
        # Función auxiliar para interpretar efecto
        def interpretar_efecto_ausentismo(corr, variable):
            if abs(corr) < 0.1:
                return "Efecto mínimo"
            elif corr > 0:
                return f"Aumenta ausentismo"
            else:
                return f"Disminuye ausentismo"
        
        # Añadir anotaciones de efecto
        for i, (_, row) in enumerate(df_plot.iterrows()):
            effect_text = interpretar_efecto_ausentismo(row['Correlacion_Spearman'], row['Variable'])
            color = 'darkred' if row['Correlacion_Spearman'] > 0 else 'darkblue'
            plt.annotate(effect_text, 
                        xy=(row['Correlacion_Spearman'], i),
                        xytext=(5, 0), textcoords='offset points',
                        ha='left', va='center', fontsize=9, color=color,
                        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', 
                        facecolor='lightyellow', alpha=0.7))
        
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        
        # Guardar gráfico
        plt.savefig(f'{self.output_path}/5_grafico_correlaciones_impacto.png', 
                    dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Gráfico de correlaciones de impacto guardado")
    
    def _clasificar_correlacion(self, corr):
        """Método auxiliar para clasificar correlación (para uso interno)"""
        abs_corr = abs(corr)
        if abs_corr < 0.1:
            return "Muy débil"
        elif abs_corr < 0.3:
            return "Débil"
        elif abs_corr < 0.5:
            return "Moderada"
        elif abs_corr < 0.7:
            return "Fuerte"
        else:
            return "Muy fuerte"
    
    def generar_mapa_calor_impacto(self, df_results):
        """Mapa de calor de impacto"""
        print("📊 Generando mapa de calor de impacto...")
        
        if df_results.empty:
            print("⚠ No hay resultados para generar mapa de calor")
            return
            
        # Crear matriz para heatmap
        heatmap_data = df_results[['Variable', 'Fuerza_Num', 'Correlacion_Spearman']].copy()
        
        # Asegurar que tenemos la columna Fuerza_Num
        if 'Fuerza_Num' not in df_results.columns:
            # Recalcular si no existe
            def clasificar_correlacion_completa(corr):
                abs_corr = abs(corr)
                if abs_corr < 0.1:
                    return "Muy débil", 1
                elif abs_corr < 0.3:
                    return "Débil", 2
                elif abs_corr < 0.5:
                    return "Moderada", 3
                elif abs_corr < 0.7:
                    return "Fuerte", 4
                else:
                    return "Muy fuerte", 5
            
            resultados = heatmap_data['Correlacion_Spearman'].apply(clasificar_correlacion_completa)
            df_temp = pd.DataFrame(resultados.tolist(), columns=['Fuerza_Spearman', 'Fuerza_Num'], index=heatmap_data.index)
            heatmap_data = pd.concat([heatmap_data, df_temp], axis=1)
        
        heatmap_data['Impacto'] = heatmap_data['Correlacion_Spearman'].apply(
            lambda x: 'Positivo' if x > 0 else 'Negativo'
        )
        heatmap_data = heatmap_data.sort_values('Fuerza_Num', ascending=False)
        
        # Crear heatmap categórico
        plt.figure(figsize=(12, 8))
        
        # Mapear fuerzas a colores
        color_map = {
            'Muy débil': 'lightgray',
            'Débil': 'lightblue', 
            'Moderada': 'orange',
            'Fuerte': 'red',
            'Muy fuerte': 'darkred'
        }
        
        # Crear matriz de colores
        colors_heatmap = []
        for _, row in heatmap_data.iterrows():
            # Usar el método interno para clasificar
            fuerza_text = self._clasificar_correlacion(row['Correlacion_Spearman'])
            if row['Impacto'] == 'Positivo':
                colors_heatmap.append(color_map.get(fuerza_text, 'gray'))
            else:
                # Para impacto negativo, usar tonos azules
                if fuerza_text == 'Muy débil': 
                    colors_heatmap.append('lightgray')
                elif fuerza_text == 'Débil': 
                    colors_heatmap.append('lightsteelblue')
                elif fuerza_text == 'Moderada': 
                    colors_heatmap.append('royalblue')
                elif fuerza_text == 'Fuerte': 
                    colors_heatmap.append('mediumblue')
                else: 
                    colors_heatmap.append('darkblue')
        
        # Crear scatter plot como heatmap
        scatter = plt.scatter(x=range(len(heatmap_data)), 
                             y=[1]*len(heatmap_data),
                             c=colors_heatmap, 
                             s=1000,  # Tamaño de los puntos
                             alpha=0.7)
        
        # Etiquetas
        plt.yticks([1], [''])
        plt.xticks(range(len(heatmap_data)), heatmap_data['Variable'], rotation=45, ha='right')
        
        # Añadir valores de correlación
        for i, (_, row) in enumerate(heatmap_data.iterrows()):
            plt.text(i, 1, f'{row["Correlacion_Spearman"]:.3f}', 
                    ha='center', va='center', fontweight='bold', fontsize=10,
                    color='white' if abs(row['Correlacion_Spearman']) > 0.3 else 'black')
        
        plt.title('MAPA DE IMPACTO - CORRELACIÓN CON AUSENTISMO\n(Tamaño del efecto y dirección)', 
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Variables')
        
        # Leyenda
        legend_elements = [
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='darkred', 
                      markersize=10, label='Muy fuerte (+)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                      markersize=10, label='Fuerte (+)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', 
                      markersize=10, label='Moderada (+)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue', 
                      markersize=10, label='Débil (+)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='darkblue', 
                      markersize=10, label='Muy fuerte (-)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='mediumblue', 
                      markersize=10, label='Fuerte (-)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='royalblue', 
                      markersize=10, label='Moderada (-)'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightsteelblue', 
                      markersize=10, label='Débil (-)')
        ]
        
        plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        
        plt.savefig(f'{self.output_path}/6_mapa_calor_impacto.png', 
                    dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Mapa de calor de impacto guardado")
    
    def generar_todas_visualizaciones(self, df, df_results_numeric, df_results_categorical, variables_numericas, target):
        """Generar todas las visualizaciones en secuencia"""
        print("\n🎨 GENERANDO TODAS LAS VISUALIZACIONES...")
        print("-" * 50)
        
        # 1. Heatmap de correlaciones
        self.generar_heatmap_correlaciones(df, variables_numericas, target)
        
        # 2. Gráfico de importancia
        self.generar_importancia_variables(df_results_numeric)
        
        # 3. Scatter plots de variables significativas
        self.generar_scatter_variables_significativas(df, df_results_numeric, target)
        
        # 4. Boxplots de variables categóricas
        self.generar_boxplots_categoricas(df, df_results_categorical, target)
        
        # 5. Gráfico de correlaciones de impacto
        self.generar_grafico_correlaciones_impacto(df_results_numeric)
        
        # 6. Mapa de calor de impacto
        self.generar_mapa_calor_impacto(df_results_numeric)
        
        print("\n✅ TODAS LAS VISUALIZACIONES GENERADAS EXITOSAMENTE!")

# Función auxiliar para clasificar correlación (necesaria para algunas visualizaciones)
def clasificar_correlacion(corr):
    """Clasificar la fuerza de la correlación"""
    abs_corr = abs(corr)
    if abs_corr < 0.1:
        return "Muy débil", 1
    elif abs_corr < 0.3:
        return "Débil", 2
    elif abs_corr < 0.5:
        return "Moderada", 3
    elif abs_corr < 0.7:
        return "Fuerte", 4
    else:
        return "Muy fuerte", 5

# CORRECCIÓN: Asegurar que df_results_numeric tenga la columna Fuerza_Num (FORMA CORREGIDA)
if 'Fuerza_Num' not in df_results_numeric.columns:
    # Aplicar la función y convertir a DataFrame
    resultados = df_results_numeric['Correlacion_Spearman'].apply(clasificar_correlacion)
    df_temp = pd.DataFrame(resultados.tolist(), columns=['Fuerza_Spearman', 'Fuerza_Num'], index=df_results_numeric.index)
    
    # Asignar las nuevas columnas al DataFrame original
    df_results_numeric = pd.concat([df_results_numeric, df_temp], axis=1)

# Generar todas las visualizaciones
viz = VisualizacionGenerator(config.OUTPUT_PATH)
viz.generar_todas_visualizaciones(
    df_clean, 
    df_results_numeric, 
    df_results_categorical, 
    VARIABLES_NUMERICAS, 
    TARGET_CONTINUO
)

print(f"\n📁 Visualizaciones guardadas en: {config.OUTPUT_PATH}")
print("✓ 1_heatmap_correlaciones.png")
print("✓ 2_importancia_variables.png") 
print("✓ 3_scatter_variables_significativas.png")
print("✓ 4_boxplots_categoricas.png")
print("✓ 5_grafico_correlaciones_impacto.png")
print("✓ 6_mapa_calor_impacto.png")


FASE 5: GENERACIÓN DE VISUALIZACIONES

🎨 GENERANDO TODAS LAS VISUALIZACIONES...
--------------------------------------------------
📊 Generando heatmap de correlaciones...
✓ Heatmap de correlaciones guardado
📊 Generando gráfico de importancia...
✓ Gráfico de importancia de variables guardado
📊 Generando scatter plots de variables significativas...
✓ Scatter plots de variables significativas guardados
📊 Generando boxplots de variables categóricas...
✓ Boxplots de variables categóricas guardados
📊 Generando gráfico de correlaciones de impacto...
✓ Gráfico de correlaciones de impacto guardado
📊 Generando mapa de calor de impacto...
✓ Mapa de calor de impacto guardado

✅ TODAS LAS VISUALIZACIONES GENERADAS EXITOSAMENTE!

📁 Visualizaciones guardadas en: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Data\Resultados_Analisis_Frecuencia_Completo_220925\Resultados_Analisis_Completo_20250926_195824
✓ 1_heatmap_correlaciones.png
✓ 2_importancia_variables.png
✓ 3_scatter_

9. Exportación de Resultados

In [44]:

print("\n" + "="*80)
print("FASE 6: EXPORTACIÓN DE RESULTADOS")
print("="*80)

class ExportadorResultados:
    """Clase para exportar resultados de manera organizada"""
    
    def __init__(self, output_path):
        self.output_path = output_path
    
    def exportar_resultados_completos(self, df_numeric, df_categorical, df_normality, df_clean):
        """Exportar todos los resultados en formatos múltiples"""
        print("💾 Exportando resultados completos...")
        
        try:
            # Crear directorio si no existe
            os.makedirs(self.output_path, exist_ok=True)
            
            # Exportar DataFrames individuales
            df_numeric.to_csv(f'{self.output_path}/resultados_numericos.csv', index=False, encoding='utf-8-sig')
            df_categorical.to_csv(f'{self.output_path}/resultados_categoricos.csv', index=False, encoding='utf-8-sig')
            df_normality.to_csv(f'{self.output_path}/analisis_normalidad.csv', index=False)
            df_clean.to_parquet(f'{self.output_path}/dataset_analisis.parquet', index=False)
            
            # Crear reporte consolidado
            self._crear_reporte_consolidado(df_numeric, df_categorical)
            
            print("✓ Todos los archivos exportados correctamente")
            
        except Exception as e:
            print(f"✗ Error en exportación: {e}")
    
    def _crear_reporte_consolidado(self, df_numeric, df_categorical):
        """Crear reporte ejecutivo consolidado"""
        print("📋 Generando reporte ejecutivo...")
        
        # Variables significativas
        vars_significativas = df_numeric[df_numeric['Significativa_Spearman']]
        vars_positivas = vars_significativas[vars_significativas['Correlacion_Spearman'] > 0]
        vars_negativas = vars_significativas[vars_significativas['Correlacion_Spearman'] < 0]
        
        with open(f'{self.output_path}/REPORTE_EJECUTIVO.txt', 'w', encoding='utf-8') as f:
            f.write("REPORTE EJECUTIVO - ANÁLISIS DE AUSENTISMO LABORAL\\n")
            f.write("="*50 + "\\n\\n")
            
            f.write("VARIABLES SIGNIFICATIVAS QUE AUMENTAN AUSENTISMO:\\n")
            for _, row in vars_positivas.iterrows():
                f.write(f"✓ {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})\\n")
            
            f.write("\\nVARIABLES SIGNIFICATIVAS QUE DISMINUYEN AUSENTISMO:\\n")
            for _, row in vars_negativas.iterrows():
                f.write(f"✓ {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})\\n")

# Exportar resultados
exportador = ExportadorResultados(config.OUTPUT_PATH)
exportador.exportar_resultados_completos(df_results_numeric, df_results_categorical, df_normality, df_clean)


FASE 6: EXPORTACIÓN DE RESULTADOS
💾 Exportando resultados completos...
📋 Generando reporte ejecutivo...
✓ Todos los archivos exportados correctamente


10. Reporte Final y Métricas

In [30]:

print("\n" + "="*80)
print("ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO")
print("="*80)

# Métricas finales
vars_analizadas = len(VARIABLES_NUMERICAS) + len(VARIABLES_CATEGORICAS)
vars_significativas = len(df_results_numeric[df_results_numeric['Significativa_Spearman']])

print(f"📊 MÉTRICAS DEL ANÁLISIS:")
print(f"• Variables analizadas: {vars_analizadas}")
print(f"• Variables numéricas significativas: {vars_significativas}")
print(f"• Variables categóricas significativas: {df_results_categorical['Significativa_Principal'].sum()}")
print(f"• Ruta de resultados: {config.OUTPUT_PATH}")

# Variables más importantes
if not df_results_numeric.empty:
    top_variables = df_results_numeric.nlargest(3, 'abs_corr')
    print(f"\\n🏆 VARIABLES MÁS IMPORTANTES:")
    for i, (_, row) in enumerate(top_variables.iterrows(), 1):
        print(f"{i}. {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})")

print(f"\\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE!")
print(f"📁 Resultados guardados en: {config.OUTPUT_PATH}")


ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO
📊 MÉTRICAS DEL ANÁLISIS:
• Variables analizadas: 23
• Variables numéricas significativas: 5
• Variables categóricas significativas: 4
• Ruta de resultados: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Data\Resultados_Analisis_Frecuencia_Completo_220925\Resultados_Analisis_Completo_20250926_190323
\n🏆 VARIABLES MÁS IMPORTANTES:
1. Disciplinary_failure: ρ = -0.390 (Fuerza_Spearman    Moderada
Fuerza_Spearman    Moderada
Name: 6, dtype: object)
2. Transportation_expense: ρ = 0.170 (Fuerza_Spearman    Débil
Fuerza_Spearman    Débil
Name: 0, dtype: object)
3. Son: ρ = 0.132 (Fuerza_Spearman    Débil
Fuerza_Spearman    Débil
Name: 7, dtype: object)
\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE!
📁 Resultados guardados en: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Data\Resultados_Analisis_Frecuencia_Completo_220925\Resultados_Analisis_Completo_20250926_190323
